# Prompt Evaluator

Score a system prompt against a dataset of queries and expected responses.

You provide:
- **`seed_prompt`** — the system prompt you want to evaluate
- **`dataset`** — queries with their expected correct responses (`ground_truth_assistant`)
- **`objective`** — the business goal the prompt is trying to achieve

The metric runs each query through your prompt, evaluates the generated response against the ground truth using an LLM judge, and returns a `prompt_score` (0.0–1.0) per session.

## How it works

For each interaction, the metric:
1. Runs `seed_prompt + context + query` through the model → generates a response
2. The judge receives the objective, query, context, expected response, and generated response
3. Reasons step by step about what is correct and what is wrong
4. Assigns a score from 0.0 to 1.0

Two metrics are reported per session:
- **`prompt_score`** — average quality across all interactions (0.0–1.0)
- **`pass_rate`** — fraction of interactions that scored above the threshold

> **Note on judge model:** You can use any LangChain-compatible model as the judge (OpenAI, Anthropic, Ollama, etc.). Keep in mind that scores will vary depending on the model you choose — each judge has its own interpretation of quality. For consistent comparisons, always use the same judge model across evaluations.

In [ ]:
!pip install alquimia-fair-forge langchain-groq -q

In [1]:
import getpass
import json
from pathlib import Path

from langchain_groq import ChatGroq

from fair_forge import Retriever
from fair_forge.metrics.prompt_evaluator import PromptEvaluator
from fair_forge.schemas import Dataset

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Configuration

Define the prompt you want to evaluate and the objective the judge will use to score responses.

In [2]:
GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

# The prompt being evaluated
SEED_PROMPT = "You are a helpful customer support assistant. Answer questions using only the information provided to you. If the answer is not in the context, say you don't have that information."

# The business goal the prompt is trying to achieve — the judge uses this as context to score responses
OBJECTIVE = "Answer the user's question accurately using only the information provided in the context, without hallucinating or adding details not present in the context."

# Interactions scoring at or above this threshold are considered passing
THRESHOLD = 0.6

## Dataset

Load your dataset. Each entry needs:
- `query` — the user question
- `context` — information available to the assistant
- `ground_truth_assistant` — the expected correct response

The `assistant` field is not used — responses are generated by the metric at runtime using the `seed_prompt`.

In [3]:
class MyRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path("dataset.json"), encoding="utf-8") as f:
            return [Dataset.model_validate(item) for item in json.load(f)]

## Run the Evaluation

In [4]:
judge = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.0)

results = PromptEvaluator.run(
    MyRetriever,
    model=judge,
    seed_prompt=SEED_PROMPT,
    objective=OBJECTIVE,
    threshold=THRESHOLD,
    verbose=True,
)

2026-03-18 13:20:28,532 - fair_forge.utils.logging - INFO - Starting to process dataset
2026-03-18 13:20:28,532 - fair_forge.utils.logging - INFO - Processing using dataset/session level parsing
2026-03-18 13:20:28,532 - fair_forge.utils.logging - INFO - Session ID: support-billing, Assistant ID: support-bot-v1
2026-03-18 13:20:28,532 - fair_forge.utils.logging - DEBUG - QA ID: billing-001
2026-03-18 13:20:28,841 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:29,205 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:29,209 - fair_forge.utils.logging - DEBUG - Score: 1.0000
2026-03-18 13:20:29,211 - fair_forge.utils.logging - DEBUG - QA ID: billing-002
2026-03-18 13:20:29,725 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:30,178 - httpx - INFO - HTTP Request: POST https://api.gr

## Results

In [5]:
for r in results:
    r.display()

Session: support-billing  |  Assistant: support-bot-v1
Prompt score:  0.94
Pass rate:     100%  (6/6 interactions passed @ threshold=0.6)

  billing-001  score=1.00  pass
  billing-002  score=0.90  pass
  billing-003  score=0.95  pass
  billing-004  score=1.00  pass
  billing-005  score=1.00  pass
  billing-006  score=0.80  pass

Session: support-auth  |  Assistant: support-bot-v1
Prompt score:  0.95
Pass rate:     100%  (5/5 interactions passed @ threshold=0.6)

  auth-001  score=0.95  pass
  auth-002  score=1.00  pass
  auth-003  score=1.00  pass
  auth-004  score=0.80  pass
  auth-005  score=1.00  pass



## Comparing Against a Weak Prompt

To verify the metric captures prompt quality, run the same dataset with a deliberately weak prompt — one that gives the model no instructions about how to use the context or avoid hallucination.

A good prompt should score noticeably higher than a weak one.

In [6]:
WEAK_PROMPT = (
    "You are a knowledgeable customer support expert. "
    "Use your full knowledge to provide thorough, comprehensive answers. "
    "If the provided information is incomplete, supplement it with your general knowledge "
    "to give the user the most complete answer possible."
)

In [7]:
weak_results = PromptEvaluator.run(
    MyRetriever,
    model=judge,
    seed_prompt=WEAK_PROMPT,
    objective=OBJECTIVE,
    threshold=THRESHOLD,
    verbose=True,
)

2026-03-18 13:20:37,563 - fair_forge.utils.logging - INFO - Starting to process dataset
2026-03-18 13:20:37,564 - fair_forge.utils.logging - INFO - Processing using dataset/session level parsing
2026-03-18 13:20:37,565 - fair_forge.utils.logging - INFO - Session ID: support-billing, Assistant ID: support-bot-v1
2026-03-18 13:20:37,565 - fair_forge.utils.logging - DEBUG - QA ID: billing-001
2026-03-18 13:20:38,051 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:38,571 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:38,574 - fair_forge.utils.logging - DEBUG - Score: 0.8000
2026-03-18 13:20:38,575 - fair_forge.utils.logging - DEBUG - QA ID: billing-002
2026-03-18 13:20:39,494 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 13:20:39,901 - httpx - INFO - HTTP Request: POST https://api.gr

## Comparison

In [8]:
print("=" * 50)
print("GOOD PROMPT")
print("=" * 50)
for r in results:
    r.display()

print()
print("=" * 50)
print("WEAK PROMPT")
print("=" * 50)
for r in weak_results:
    r.display()

GOOD PROMPT
Session: support-billing  |  Assistant: support-bot-v1
Prompt score:  0.94
Pass rate:     100%  (6/6 interactions passed @ threshold=0.6)

  billing-001  score=1.00  pass
  billing-002  score=0.90  pass
  billing-003  score=0.95  pass
  billing-004  score=1.00  pass
  billing-005  score=1.00  pass
  billing-006  score=0.80  pass

Session: support-auth  |  Assistant: support-bot-v1
Prompt score:  0.95
Pass rate:     100%  (5/5 interactions passed @ threshold=0.6)

  auth-001  score=0.95  pass
  auth-002  score=1.00  pass
  auth-003  score=1.00  pass
  auth-004  score=0.80  pass
  auth-005  score=1.00  pass


WEAK PROMPT
Session: support-billing  |  Assistant: support-bot-v1
Prompt score:  0.27
Pass rate:     33%  (2/6 interactions passed @ threshold=0.6)

  billing-001  score=0.80  pass
  billing-002  score=0.00  FAIL
  billing-003  score=0.20  FAIL
  billing-004  score=0.60  pass
  billing-005  score=0.00  FAIL
  billing-006  score=0.00  FAIL

Session: support-auth  |  Assi